# Chapter 7 · Variational Quantum Eigensolver (VQE)

## Objectives

1. Understand the quantum variational principle and its implementation on noisy hardware.
2. Build a parametric ansatz (RealAmplitudes / EfficientSU2).
3. Compute the expected value of a Pauli Hamiltonian with Qiskit.
4. Optimize the parameters with SPSA and COBYLA to find the ground state energy.

---

## 7.1 Variational principle

For any state $|\psi(\boldsymbol{\theta})\rangle$ and Hamiltonian $H$:

$$E_0 \leq \langle\psi(\boldsymbol{\theta})|H|\psi(\boldsymbol{\theta})\rangle \equiv E(\boldsymbol{\theta})$$

VQE minimizes $E(\boldsymbol{\theta})$ over the parametric space $\boldsymbol{\theta} \in \mathbb{R}^m$ using a classical optimizer, obtaining an approximation to the ground state $E_0$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import RealAmplitudes, EfficientSU2
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator
from scipy.optimize import minimize

print('Qiskit and dependencies loaded.')

## 7.2 Test Hamiltonian: simplified H₂

We will use the 2-qubit Heisenberg Hamiltonian as a pedagogical example:

$$H = -J(X \otimes X + Y \otimes Y + Z \otimes Z)$$

with $J = 1$, whose ground state is one of the singlet Bell states $|\Psi^-\rangle$ with energy $E_0 = -3$.

In [ ]:
# Heisenberg Hamiltonian 2 qubits
J = 1.0
hamiltonian = SparsePauliOp.from_list([
    ('XX', -J),
    ('YY', -J),
    ('ZZ', -J),
])

# Exact energy by diagonalization
H_matrix = hamiltonian.to_matrix()
eigenvalues = np.linalg.eigvalsh(H_matrix)
E_exact = np.min(eigenvalues)

print('Heisenberg Hamiltonian (2 qubits):')
print(hamiltonian)
print(f'\nEigenvalues: {eigenvalues}')
print(f'Ground state energy E\u2080 = {E_exact:.6f}')

## 7.3 Ansatz and expected value evaluation

In [ ]:
# Ansatz: RealAmplitudes with 2 qubits and 2 layers
n_qubits = 2
n_reps   = 2
ansatz = RealAmplitudes(n_qubits, reps=n_reps, entanglement='linear')
n_params = ansatz.num_parameters

print(f'Ansatz: RealAmplitudes (n_qubits={n_qubits}, reps={n_reps})')
print(f'Number of parameters: {n_params}')
print(ansatz.decompose().draw('text'))

# Estimator to evaluate ⟨ψ(θ)|H|ψ(θ)⟩
estimator = Estimator()

## 7.4 VQE optimization loop

In [ ]:
energy_history = []

def cost_function(params: np.ndarray) -> float:
    """Computes the expected value ⟨H⟩ for the given parameters."""
    bound_circuit = ansatz.assign_parameters(params)
    sv = Statevector(bound_circuit)
    H_matrix = hamiltonian.to_matrix()
    energy = np.real(sv.data.conj() @ H_matrix @ sv.data)
    energy_history.append(energy)
    return energy

# Random initial parameters
np.random.seed(42)
theta_init = np.random.uniform(0, 2 * np.pi, n_params)

print(f'Initial energy: {cost_function(theta_init):.6f}')
print(f'Exact energy:   {E_exact:.6f}')
print('\nOptimizing with COBYLA...')

result = minimize(
    cost_function,
    theta_init,
    method='COBYLA',
    options={'maxiter': 500, 'rhobeg': 0.5},
)

E_vqe   = result.fun
theta_opt = result.x

print(f'\n=== VQE Result ===')
print(f'VQE energy      = {E_vqe:.6f}')
print(f'Exact energy    = {E_exact:.6f}')
print(f'Absolute error  = {abs(E_vqe - E_exact):.2e}')
print(f'Iterations      = {result.nfev}')

In [ ]:
# Convergence curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energy_history, color='#58a6ff', linewidth=1.5, label='E(θ) VQE')
ax.axhline(E_exact, color='#f78166', linestyle='--', linewidth=1.5,
           label=f'Exact E\u2080 = {E_exact:.4f}')
ax.set_xlabel('Optimizer iteration')
ax.set_ylabel('Energy ⟨H⟩')
ax.set_title('VQE Convergence \u2014 Heisenberg Hamiltonian 2 qubits')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 7.5 Optimal state verification

In [ ]:
# Optimal state
optimal_circuit = ansatz.assign_parameters(theta_opt)
sv_opt = Statevector(optimal_circuit)

print('Ground state approximated by VQE:')
for i, amp in enumerate(sv_opt.data):
    print(f'  |{format(i, "02b")}\u27e9: {amp:.4f}  (prob = {np.abs(amp)**2:.4f})')

print('\nExact singlet state |\u03a8-\u27e9 = (|01\u27e9 - |10\u27e9)/\u221a2:')
psi_minus_exact = np.array([0, 1/np.sqrt(2), -1/np.sqrt(2), 0])
for i, amp in enumerate(psi_minus_exact):
    print(f'  |{format(i, "02b")}\u27e9: {amp:.4f}  (prob = {np.abs(amp)**2:.4f})')

fidelidad = np.abs(np.dot(sv_opt.data.conj(), psi_minus_exact))**2
print(f'\nFidelity VQE vs |\u03a8-\u27e9 = {fidelidad:.6f}')

## 7.6 Proposed exercises

1. Repeat the VQE with the SPSA optimizer, which does not require a gradient. Compare the number of circuit evaluations.

2. Add noise to the simulator (decoherence model) and analyze how it affects the minimum energy that VQE can reach.

3. Try the `EfficientSU2` ansatz with complex parameters. Do you need more or fewer iterations to converge?

4. Implement VQE for the 1D Ising Hamiltonian with 4 qubits: $H = -J\sum_i Z_i Z_{i+1} - h\sum_i X_i$.